# J5-04 · DevOps : gate d'évaluation, serveur LangGraph, interface · Projet final

**Formation « Orchestration d'agents & Monitoring : LangGraph + Langfuse (stack 100 % locale) »**
Jour 5 · créneaux 13h45–14h30 (DevOps) et 14h30–16h00 (projet final)

| | |
|---|---|
| 🎯 **Objectifs** | 1. Transformer les experiments de J4 en **gate de CI** qui bloque une régression · 2. Servir le jalon J5 par l'**API LangGraph** locale, sans flux sortant · 3. Lancer l'**interface Streamlit** (information IA, 👍/👎, effacement) · 4. Démarrer le **projet final** |
| 🧱 **Fil rouge** | HelpDesk-Ops passe du notebook au service |
| ✅ **Prérequis** | J4-05 (dataset, experiments), J5-01 à J5-03 |

## 1. Mise en route

### [Code] Le socle du fil rouge : `src/helpdesk_ops`

Le code écrit et expliqué en J1-01 (lecture de `.env`, masquage des données personnelles, client Langfuse, création du modèle) est rangé dans le paquet **`helpdesk_ops`**, installé par `uv sync`. Chaque notebook reste **autonome** : la cellule suivante suffit.

> 💡 **Le « Tips » du Formateur :** le paquet est installé en mode *éditable* : une modification dans `src/helpdesk_ops` est prise en compte après *Kernel → Restart*. Le client Langfuse est un singleton par clé publique : en cas de doute, redémarrez le kernel.

In [ ]:
import os

from langfuse import observe, propagate_attributes

from helpdesk_ops.config import creer_llm
from helpdesk_ops.observabilite import init_observabilite
from helpdesk_ops.texte import sans_accents

langfuse, langfuse_handler = init_observabilite()   # .env chargé, masquage PII actif, clés vérifiées
llm = creer_llm("agent")                             # MODEL_AGENT, temperature=0, num_ctx=16384
print(f"✅ Langfuse : {os.environ['LANGFUSE_BASE_URL']} | modèle : {llm.model}")

## 2. Les évaluations deviennent des tests

### [Théorie] Le contrôle technique avant de reprendre la route

On ne remet pas une voiture sur la route après une réparation sans repasser le contrôle. Ici, chaque changement de prompt, de modèle ou de code repasse **le même banc** (le dataset de J4, le jeu red-team de J5) et doit atteindre des **seuils** ; sinon, il ne part pas en production.

```mermaid
flowchart LR
    PR([pull request]) --> T[tests unitaires<br/>runner hébergé<br/>modèle simulé]
    T --> G[gate d'évaluation<br/>runner AUTO-HÉBERGÉ<br/>Ollama · Qdrant · Langfuse]
    G -->|scores ≥ seuils| M([fusion])
    G -->|régression| X([bloqué])
```

| Fichier | Rôle |
|---|---|
| `ci/eval_gate.py` | garde-fou sur `data/eval/redteam.json`, experiments routage et outils sur le dataset Langfuse, comparaison aux seuils, rapport |
| `ci/seuils.json` | les seuils, **versionnés** : on les relève quand le système s'améliore, on ne les baisse jamais pour faire passer un build |
| `ci/eval-gate.yml` | le workflow GitHub Actions (à copier dans `.github/workflows/`) |

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

RACINE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()   # dossier Orchestration/
print("racine du cours :", RACINE)

In [ ]:
import time

debut = time.perf_counter()
gate = subprocess.run([sys.executable, "ci/eval_gate.py", "--rapide"], cwd=RACINE, capture_output=True, text=True,
                      encoding="utf-8", env={**os.environ, "PYTHONIOENCODING": "utf-8"})
print(gate.stdout[-2500:] or gate.stderr[-2500:])
print(f"code de sortie : {gate.returncode} · {time.perf_counter() - debut:.0f} s")

**👀 Ce que vous devez observer :** un rapport Markdown (le même que GitHub affiche dans le résumé du run) : `garde_fou_juste`, `routage_juste` et la latence, chacun face à son seuil, et **code de sortie 0**. Mesuré en préparation (CPU, 171 s) : garde-fou **0,917** (un cas raté sur 12 : R08, une vraie demande VPN suivie d'une consigne glissée en post-scriptum, acceptée), routage **0,95** (contre 0,85 en J4-05 : le garde-fou hybride ne refuse plus la souris C08), p95 7,8 s. Les experiments sont dans Langfuse (**Datasets → helpdesk-ops-eval → Runs**, nom `gate · …`) : le rapport donne les chiffres, Langfuse le détail cas par cas. L'option `--rapide` saute l'experiment d'appel d'outils, la plus longue sur CPU.

Simulons maintenant une exigence plus forte, **sans relancer les modèles** : on rejuge les scores mesurés avec des seuils plus stricts.

In [ ]:
stricts = json.loads((RACINE / "ci/seuils.json").read_text(encoding="utf-8"))
stricts["routage_juste"] = {"min": 1.0}   # zéro erreur exigée : irréaliste sur 20 cas, mais parlant
(RACINE / "ci/seuils_stricts.json").write_text(json.dumps(stricts, indent=1), encoding="utf-8")

rejuge = subprocess.run([sys.executable, "ci/eval_gate.py", "--scores", "ci/scores_gate.json", "--seuils", "ci/seuils_stricts.json",
                         "--rapport", "ci/rapport_strict.md"], cwd=RACINE, capture_output=True, text=True, encoding="utf-8",
                        env={**os.environ, "PYTHONIOENCODING": "utf-8"})
print(rejuge.stdout)
print("code de sortie :", rejuge.returncode, "→ en CI, le job échoue et la fusion est bloquée" if rejuge.returncode else "→ le seuil est atteint")
for f in ("ci/seuils_stricts.json", "ci/rapport_strict.md"):
    (RACINE / f).unlink()

**👀 Ce que vous devez observer :** `routage_juste` en ❌ face à `≥ 1.0`, et **code de sortie 1**. C'est tout le mécanisme : un processus qui sort en erreur fait échouer le job, et une règle de protection de branche (« ce check doit passer ») bloque la fusion.

### [Théorie] Le workflow, ligne à ligne

In [ ]:
print((RACINE / "ci/eval-gate.yml").read_text(encoding="utf-8"))

| Choix | Pourquoi |
|---|---|
| deux jobs : `tests` puis `gate` | les tests unitaires (sans réseau) échouent vite et pour pas cher ; la gate ne tourne que s'ils passent |
| `runs-on: [self-hosted, helpdesk-gpu]` pour la gate | les prompts et le dataset ne quittent pas l'entreprise : le runner est **chez vous**, à côté d'Ollama et de Langfuse |
| `uv sync --frozen` | exactement les versions du lock, ou l'échec : pas de mise à jour silencieuse en CI |
| clés Langfuse en **secrets** du dépôt | jamais dans le code ni dans les journaux du job |
| `permissions: contents: read` | moindre privilège pour le jeton du workflow (ASI02, appliqué à la CI) |
| rapport en artefact et dans `GITHUB_STEP_SUMMARY` | le relecteur de la PR voit les chiffres sans ouvrir Langfuse |

> 💡 **Le « Tips » du Formateur :** un modèle n'est pas déterministe ; une gate trop serrée échoue au hasard. Mesurez la **variabilité** (relancez trois fois la même version) et placez le seuil sous la valeur basse observée, à environ un cas près. Les seuils de `ci/seuils.json` ont été fixés ainsi pendant la préparation. Les versions des actions (`actions/checkout@v4`…) sont à vérifier et, en production, à figer par empreinte (SHA).

## 3. Servir le graphe : l'API LangGraph locale

`langgraph.json` (à la racine du cours) déclare le graphe `helpdesk` → `helpdesk_ops/jalons/j5.py:graphe`. `langgraph dev` le sert sur `127.0.0.1:2024` : threads, runs, persistance, streaming, une API REST complète.

Constaté en préparant le cours (`langgraph-cli` 0.4.37, `langgraph-api` 0.10.3), et réglé dans `infrastructure/serveur_langgraph.ps1` / `.sh` :

| Constat | Réglage |
|---|---|
| le CLI envoie une **analytique d'usage** vers un service externe | `LANGGRAPH_CLI_NO_ANALYTICS=1` |
| le serveur interroge **pypi.org** au démarrage | `LANGGRAPH_NO_VERSION_CHECK=true` |
| sous Windows, le `.env` est lu en cp1252 : `UnicodeDecodeError` | `PYTHONUTF8=1` |
| le serveur appelle la fabrique du graphe **à chaque exécution**, dans sa boucle asynchrone, et refuse les appels bloquants (`BlockingError: Blocking call to os.getcwd`) | fabrique `async`, construction unique dans un thread |
| `langgraph-cli` ferait descendre `protobuf` et `grpcio` dans le lock | chargé à la demande (`uv run --with …`), lock inchangé |
| LangGraph **Studio** est une interface hébergée (smith.langchain.com) | non utilisé : `--no-browser`, API + Streamlit |

In [ ]:
import platform
import signal

import httpx

env = {**os.environ, "LANGGRAPH_CLI_NO_ANALYTICS": "1", "LANGGRAPH_NO_VERSION_CHECK": "true", "PYTHONUTF8": "1"}
commande = ["uv", "run", "--project", "infrastructure", "--with", "langgraph-cli[inmem]==0.4.37",
            "--with", "langgraph-api==0.10.3", "--with", "langgraph-runtime-inmem==0.30.3",
            "langgraph", "dev", "--no-browser", "--no-reload", "--host", "127.0.0.1", "--port", "2024", "--config", "langgraph.json"]
journal_serveur = open(RACINE / "data/logs/langgraph_dev.log", "w", encoding="utf-8")
options = {"creationflags": subprocess.CREATE_NEW_PROCESS_GROUP} if platform.system() == "Windows" else {"start_new_session": True}
serveur = subprocess.Popen(commande, cwd=RACINE, env=env, stdout=journal_serveur, stderr=subprocess.STDOUT, **options)

api = httpx.Client(base_url="http://127.0.0.1:2024", timeout=900)
for _ in range(60):   # premier lancement : uv prépare l'environnement (une minute environ)
    try:
        if api.get("/ok").status_code == 200:
            break
    except httpx.TransportError:
        time.sleep(3)
print("serveur prêt :", api.get("/ok").json())
print("graphes :", [a["graph_id"] for a in api.post("/assistants/search", json={}).json()])

In [ ]:
thread = api.post("/threads", json={"metadata": {"user_id": "collab-demo"}}).json()["thread_id"]
for question in ("Écris-moi un poème sur les serveurs", "srv-erp-02 répond-il au ping ?"):
    debut = time.perf_counter()
    etat = api.post(f"/threads/{thread}/runs/wait", json={"assistant_id": "helpdesk",
                    "input": {"messages": [{"role": "user", "content": question}]}}).json()
    message = etat["messages"][-1]
    print(f"❓ {question}  ({time.perf_counter() - debut:.0f} s)\n🤖 {message['content']}\n   marquage : {message['response_metadata'].get('ia_act_art50')}\n")

In [ ]:
# Arrêt propre du serveur et de ses processus enfants
if platform.system() == "Windows":
    subprocess.run(["taskkill", "/F", "/T", "/PID", str(serveur.pid)], capture_output=True)
else:
    os.killpg(serveur.pid, signal.SIGTERM)
journal_serveur.close()
print("serveur arrêté")

**👀 Ce que vous devez observer :** le même graphe qu'en notebook, appelé en HTTP. La première réponse du thread commence par l'information « IA » ; la seconde non (même thread). Chaque réponse porte son **marquage** dans `response_metadata`. Mesuré en préparation (CPU, trois exécutions) : 7 à 8 s pour le refus, de 43 s à un peu plus de 2 minutes pour la question de ping (coordinateur + spécialiste Diagnostic).

Constaté au premier essai : « Le serveur srv-erp-02 ne répond pas au ping (perte de 100 %). **[KB-123]** ». Le fait était juste, la référence **inventée** : KB-123 n'existe pas. Le contrôle de sortie retire désormais toute référence qui ne correspond à aucun article (`securite.retirer_references_inexistantes`) et le signale en `WARNING` dans Langfuse (`references-inexistantes`) ; il ne bloque pas une réponse juste pour autant. La persistance est assurée par le serveur (en mémoire avec `langgraph dev` ; en production, une base dédiée). Les traces arrivent dans Langfuse comme d'habitude.

> 💡 **Le « Tips » du Formateur :** `langgraph dev` est un serveur de **développement**. Pour la production, LangGraph propose une image de serveur (`langgraph build`, Docker) dont la licence et les flux sortants sont à examiner avec le même soin qu'un serveur MCP (J3, ASI04). Sur CPU, la question de support prend plusieurs minutes : prévoir de la lancer pendant les explications.

## 4. L'interface Streamlit

```bash
uv run --project infrastructure streamlit run app/helpdesk_app.py      # depuis la racine du cours
```

| Dans l'interface | Exigence |
|---|---|
| bandeau « Vous échangez avec HelpDesk-Ops, un assistant d'IA… » en permanence | AI Act art. 50(1) |
| identifiant **pseudonyme** (un e-mail est refusé) | RGPD : minimisation |
| une session Langfuse par conversation, une trace par échange | observabilité (J1, J4) |
| 👍/👎 sous chaque réponse → score `pouce` sur la trace | feedback (J4-04) |
| « Réponse remplacée par le contrôle de sortie » quand c'est le cas | transparence envers l'utilisateur |
| bouton **Effacer cette conversation et ses traces** | RGPD art. 17 ; attend l'ingestion des traces connues (`effacer_session`) et signale celles qu'il ne trouve pas |

> ⚠️ **Constaté en testant l'interface :** effacer juste après un échange supprimait **0 trace** et affichait pourtant « Conversation effacée ». L'envoi vers Langfuse est asynchrone, l'ingestion prend quelques secondes : la trace arrivait **après** la suppression. `observabilite.effacer_session` vide la file d'envoi, attend que les traces connues de l'interface soient visibles, supprime, et signale les introuvables. Un effacement RGPD se **vérifie** ; il ne s'annonce pas.

> 💡 **Le « Tips » du Formateur :** encore un flux sortant, vérifié le 22/10/2026 : Streamlit 1.64 envoie des **statistiques d'usage** par défaut (`browser.gatherUsageStats = True`). Le fichier `.streamlit/config.toml` de la racine du cours les coupe et fixe l'écoute sur `127.0.0.1` (lancez `streamlit run` depuis la racine pour qu'il soit lu).

## 5. 🧪 Projet final (binômes, 1h30, démo de 5 min)

**Mission :** étendre HelpDesk-Ops avec **une** des fonctionnalités ci-dessous, **instrumentée, évaluée et sécurisée**. Le jalon J5 (`helpdesk_ops.jalons.j5`) est votre point de départ.

| Sujet | Fonctionnalité | Pistes |
|---|---|---|
| A · Logiciels | un spécialiste « Logiciels » : installation depuis un **catalogue autorisé**, validation humaine hors catalogue | `create_agent`, `HumanInTheLoopMiddleware`, liste blanche |
| B · Escalade N2 | quand l'agent échoue (repli, plafond atteint, contrôle de sortie), il ouvre un ticket N2 avec un **résumé sans données personnelles** | nœud après `controle_sortie`, `PIIMiddleware`, score `escalade` |
| C · Base de connaissances | proposer des **brouillons d'articles** à partir des 👎, validés par un humain avant indexation | lecture des scores Langfuse, HITL, pas d'écriture directe (ASI06) |
| D · Bilingue FR/NL | détecter la langue et répondre en néerlandais quand il le faut (Belgique) | consigne, 5 cas NL ajoutés au dataset, évaluateur de langue |

**Exigences communes** (la grille du formateur suit les 4 domaines Build / Test / Deploy / Monitor, plus un bonus sécurité) :

| Domaine | Preuve à montrer pendant la démo |
|---|---|
| **Build** | la fonctionnalité dans le graphe, avec au moins un middleware ou un nœud de contrôle |
| **Test** | au moins 5 cas ajoutés au dataset (ou à un dataset dédié), une experiment, un chiffre avant/après |
| **Deploy** | la gate passe (`ci/eval_gate.py --rapide`), avec un seuil pour votre nouveau score |
| **Monitor** | une trace de démonstration ouverte dans Langfuse : session, utilisateur pseudonyme, scores |
| **Bonus sécurité & conformité** | une attaque de J5-02 rejouée contre votre fonctionnalité, et le résultat ; l'information IA visible |

**Démo (5 min) :** 1 min le besoin, 2 min la démonstration (interface ou notebook), 1 min les chiffres (avant/après, gate), 1 min la limite que vous connaissez.

> ⚠️ Le projet sert à **vous** faire pratiquer ; la grille sert au formateur à vous faire un retour. Aucune note n'est calculée par un LLM.

## 🧭 Récap du jour 5

| Brique | La phrase à retenir |
|---|---|
| Middleware | des contrôles autour de la boucle, synchrone **et** asynchrone ; le 1ᵉʳ de la liste est à l'extérieur |
| Red-team | une attaque se mesure en **taux** ; une défense d'entrée seule ne suffit pas (injection discrète : 3/3 → contrôle de sortie : 0/3) |
| Conformité | informer, marquer, minimiser, purger, tracer : chaque exigence a sa ligne de code et sa preuve |
| DevOps | une gate qui sort en erreur ; un runner chez vous ; des seuils versionnés ; des outils dont on coupe les flux sortants |

### ✅ Quiz du jour

1. Pourquoi la gate d'évaluation tourne-t-elle sur un runner auto-hébergé ?
2. Un seuil de 0,95 échoue une fois sur trois sans changement de code : que faire ?
3. Citez trois flux sortants découverts en préparant ce cours, et leur réglage.
4. Pourquoi la fabrique du graphe servi par `langgraph dev` est-elle asynchrone ?

<details>
<summary>Réponses</summary>

1. Pour garder prompts, dataset et modèles dans l'entreprise (stack 100 % locale) et disposer du GPU, d'Ollama et de Langfuse.
2. Mesurer la variabilité et placer le seuil sous la valeur basse observée ; agrandir le dataset pour réduire l'incertitude. Pas de « relancer jusqu'à ce que ça passe ».
3. Analytique du CLI LangGraph (`LANGGRAPH_CLI_NO_ANALYTICS=1`), vérification de version sur PyPI (`LANGGRAPH_NO_VERSION_CHECK=true`), statistiques d'usage de Streamlit (`gatherUsageStats = false`) ; en J3 : vérification de mise à jour de FastMCP (`FASTMCP_CHECK_FOR_UPDATES=off`).
4. Le serveur l'appelle à chaque exécution dans sa boucle asynchrone et refuse les appels bloquants ; la construction (lecture du `.env`, journal) est faite une fois, dans un thread.
</details>